# Estudo de Compatibilidade de Marginais Quânticas via Misturas de Estados de Bell

Este notebook investiga o **Problema das Marginais Quânticas (Quantum Marginal Problem - QMP)** para uma família paramétrica de estados mistos de 2 qubits descritos por uma mistura do estado maximalmente emaranhado de Bell $|\Phi^+\rangle$ com o estado maximalmente misto $I/4$:

$$\rho(t) = t |\Phi^+\rangle\langle\Phi^+| + (1-t) \frac{I_4}{4}, \quad t \in [0, 1]$$

### Objetivos Teóricos e Numéricos:
1. **Verificação de Extensibilidade Global**: Determinar para quais valores do parâmetro de mistura $t$ existe uma matriz densidade global válida $\rho_{ABC} \ge 0$ de 3 qubits cujas marginais reduzidas satisfazem $\text{Tr}_C(\rho_{ABC}) = \rho_{AB}(t)$ e $\text{Tr}_B(\rho_{ABC}) = \rho_{AC}(t)$.
2. **Fronteira de Compatibilidade e Monogamia do Emaranhamento**: Identificar o limite teórico em $t = 2/3$, além do qual a monogamia do emaranhamento impede a existência de uma extensão global factível.
3. **Análise Numérica via Programação Semidefinida (SDP)**: Resolver o QMP utilizando o solver **CLARABEL** via biblioteca `SDPLab` e analisar propriedades físicas como pureza $\text{Tr}(\rho_{ABC}^2)$, menor autovalor $\lambda_{\min}$ e o espectro de autovalores de $\rho_{ABC}$.


## 1. Importação de Módulos e Dependências Fundamentais

Carregamos as bibliotecas essenciais para álgebra linear numérica (`numpy`), análise de dados (`pandas`), visualização gráfica (`matplotlib`) e o módulo interno `funcoes_fundamentais.py` contendo primitivas quânticas (traço parcial, produto de Kronecker, projetores e montagem de SDP).


In [ ]:
from funcoes_fundamentais import *
import pandas as pd
import numpy as np


### 1.1 Instalação Automatizada de Pacotes do Ambiente

Instalação dos pacotes especializados para Programação Semidefinida e informação quântica (`spacecore==0.4.2`, `sdp`).


In [ ]:
# Instalação das dependências necessárias no ambiente
# Nota: spacecore==0.4.2 é recomendado para total compatibilidade com sdplab==0.0.1
! pip install -q numpy matplotlib scipy "spacecore==0.4.2" sdplab cvxpy clarabel scs jax

### 1.2 Configurações de Tipagem e Gerenciamento de Memória

Importações auxiliares para otimização do coletor de lixo (`gc`) e suporte a anotações de tipo avançadas (`annotations`).


In [ ]:
from __future__ import annotations

import sys
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
# Bibliotecas SpaceCore e SDPLab para Programação Semidefinida 
import spacecore
from spacecore import Context, DenseVectorSpace, HermitianSpace, NumpyOps
import sdplab
from sdplab import DenseConstraintOp, SDPProblem
from sdplab.solvers import run_cvxpy_solver

np.set_printoptions(precision=4, suppress=True)

from funcoes_fundamentais import zero, one, ketbra, verifica_densidade, verify_marginals, base_hermitiana, ghz, partial_trace, tensor, embed_operator, solve_qmp, build_qmp_sdp, parse_subsystem_key, basis
print(f"Versão SDPLab:    {sdplab.__version__}")
print(f"Versão SpaceCore: {spacecore.__version__}")

## 2. Construção do Estado Maximamente Emaranhado de Bell

Definimos o estado de Bell $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ no espaço de Hilbert composto $\mathcal{H}_A \otimes \mathcal{H}_B \cong \mathbb{C}^4$.
Em seguida, geramos o operador densidade associado via produto exterior (projetor):

$$\rho_{\text{Bell}} = |\Phi^+\rangle\langle\Phi^+|$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt




phi_plus = (
    np.kron(zero, zero) +
    np.kron(one, one)
) / np.sqrt(2.0)

rho_bell = ketbra(phi_plus)




## 3. Discretização do Parâmetro de Mistura $t$ e Estrutura de Dados

Construímos uma malha fina com 101 pontos no intervalo $t \in [0, 1]$ para realizar o escaneamento numérico da família de estados:

$$\rho(t) = t \rho_{\text{Bell}} + (1-t) \frac{I_4}{4}$$

Inicializamos o dicionário `resultados` para registrar os diagnósticos da solução SDP obtida pelo solver.


In [ ]:

t_espaco = np.linspace(0, 1, 101)

resultados = {
    "t": [],
    "purity": [],
    "min_eigenvalue": [],
    "trace": [],
    "marginal_error_max": [],
    "entropy": [],
    "eigenvalues": []
}




## 4. Loop Principal: Resolução do SDP e Varredura Paramétrica

Para cada ponto $t$:
1. Definimos a restrição de marginais bipartidas idênticas: $\rho_{AB} = \rho(t)$ e $\rho_{AC} = 
ho(t)$.
2. Formulamos o SDP procurando $\rho_{ABC} \ge 0$ de dimensão $2^3 = 8$ que satisfaça $\text{Tr}_C(\rho_{ABC}) = \rho_{AB}$ e $\text{Tr}_B(\rho_{ABC}) = \rho_{AC}$.
3. Resolvemos a otimização convexa utilizando o solver **CLARABEL**.
4. Armazenamos as métricas físicas da extensão global $\rho_{ABC}$ (pureza, autovalores, erro de aproximação).


In [ ]:




for t in t_espaco:

    rho_p = mestura(
        rho_bell,
        np.eye(4) / 4,
        t
    )

    marginals_ex4 = {
        "AB": rho_p,
        "AC": rho_p
    }

    print(f"p = {t:.6f}")

    # --------------------------------------------------------
    # Construção do SDP
    # --------------------------------------------------------

    sdp_ex4, M_ex4, b_ex4 = build_qmp_sdp(
        marginals_ex4,
        dims=[2, 2, 2]
    )

    # --------------------------------------------------------
    # Resolução
    # --------------------------------------------------------

    res_ex4 = solve_qmp(
        sdp_ex4,
        solver="CLARABEL"
    )

    print(
        f"Status: {res_ex4['status']} | "
        f"Factível: {res_ex4['is_feasible']}"
    )

    # --------------------------------------------------------
    # Só armazenamos se houver solução
    # --------------------------------------------------------

    if res_ex4["is_feasible"]:

        rho_ABC = res_ex4["rho"]

        verif_ex4 = verify_marginals(
            rho_ABC,
            marginals_ex4,
            dims=[2, 2, 2]
        )

        eigenvalues = np.sort(
            np.real(
                np.linalg.eigvalsh(rho_ABC)
            )
        )

        marginal_error_max = max(
            err["frobenius_error"]
            for err in verif_ex4["marginal_errors"].values()
        )

        resultados["t"].append(t)

        resultados["purity"].append(
            verif_ex4["purity"]
        )

        resultados["min_eigenvalue"].append(
            verif_ex4["min_eigenvalue"]
        )

        resultados["trace"].append(
            verif_ex4["trace"].real
        )

        resultados["marginal_error_max"].append(
            marginal_error_max
        )


        resultados["eigenvalues"].append(
            eigenvalues
        )


# Converter para numpy
for chave in resultados:
    resultados[chave] = np.array(
        resultados[chave]
    )

## 5. Análise Visual dos Resultados do Estudo de Mistura

### 5.1 Pureza da Extensão Global $\text{Tr}(\rho_{ABC}^2)$

A pureza mede o grau de ordenação do estado global resultante. Para $t=0$, o estado é maximalmente misto (pureza $1/8 = 0.125$). Conforme $t$ aumenta até $t=2/3$, a pureza cresce até a borda da região factível.


In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    resultados["t"],
    resultados["purity"],
    "o-"
)

plt.axvline(
    2/3,
    linestyle="--",
    label=r"$t=2/3$"
)

plt.xlabel(r"$t$")
plt.ylabel(r"$\mathrm{Tr}(\rho_{ABC}^2)$")
plt.title("Pureza da extensão global")
plt.grid(alpha=0.3)
plt.legend()

plt.show()

### 5.2 Menor Autovalor da Extensão Global $\lambda_{\min}(\rho_{ABC})$

A condição de positividade $\rho_{ABC} \ge 0$ exige que todos os autovalores de $\rho_{ABC}$ sejam não-negativos. O gráfico confirma que $\lambda_{\min} \ge 0$ dentro da região factível $t \le 2/3$.


In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    resultados["t"],
    resultados["min_eigenvalue"],
    "o-"
)

plt.axhline(
    0,
    linestyle="--"
)

plt.axvline(
    2/3,
    linestyle="--",
    label=r"$t=2/3$"
)

plt.xlabel(r"$t$")
plt.ylabel(r"$\lambda_{\min}(\rho_{ABC})$")
plt.title("Menor autovalor da extensão global")
plt.grid(alpha=0.3)
plt.legend()

plt.show()

### 5.3 Erro Máximo de Reconstrução das Marginais

Mede a norma de Frobenius da diferença entre a marginal obtida pelo SDP e a marginal de entrada. O erro na faixa de $10^{-8}$ a $10^{-14}$ atesta a alta precisão do solver **CLARABEL**.


In [ ]:
plt.figure(figsize=(7, 5))

plt.semilogy(
    resultados["t"],
    resultados["marginal_error_max"],
    "o-"
)

plt.axvline(
    2/3,
    linestyle="--",
    label=r"$t=2/3$"
)

plt.xlabel(r"$t$")
plt.ylabel(r"Erro máximo $\|\rho_{ij}^{calc}-\rho_{ij}\|_F$")
plt.title("Erro de reconstrução das marginais")
plt.grid(alpha=0.3)
plt.legend()

plt.show()

### 5.4 Espectro Completo de Autovalores

Trajetória de todos os 8 autovalores ordenados do estado tripartite $\rho_{ABC}$. Conforme $t \to 2/3$, o menor autovalor se aproxima de zero até cruzar o limiar de factibilidade.


In [ ]:
plt.figure(figsize=(8, 5))

eigs = resultados["eigenvalues"]

for i in range(eigs.shape[1]):
    plt.plot(
        resultados["t"],
        eigs[:, i],
        "o-",
        label=fr"$\lambda_{i+1}$"
    )

plt.axvline(
    2/3,
    linestyle="--",
    label=r"$t=2/3$"
)
print(f'Autovalores em t = {resultados['t'][-1]}:{eigs[-1, :]}')
plt.xlabel(r"$t$")
plt.ylabel(r"Autovalores de $\rho_{ABC}$")
plt.title("Espectro da extensão global")
plt.grid(alpha=0.3)
plt.legend()

plt.show()

# Parte 2: Estudo Numérico Amplo da Região de Compatibilidade (3 Qubits)

Esta segunda seção investiga estatisticamente a relação entre duas condições cruciais em mecânica quântica:

1. **Consistência na Sobreposição (Overlap Consistency)**: $\text{Tr}_B(\rho_{AB}) = \text{Tr}_C(\rho_{AC}) = \rho_A$.
2. **Extensibilidade Global (Global Extendability)**: Existência de $\rho_{ABC} \ge 0$ que estenda simultaneamente $\rho_{AB}$ e $\rho_{AC}$.

**Resultado Fundamental**: A consistência na sobreposição é uma condição **necessária**, mas **NÃO suficiente** para a existência de um estado global compatível.


## 6. Geração de Matrizes Densidade Aleatórias (Ensemble de Ginibre)

Implementação do método de Ginibre: geramos matrizes complexas $G$ aleatórias gaussianas e definimos $\rho = G G^\dagger / \text{Tr}(G G^\dagger)$, garantindo matrizes densidade distribuídas uniformemente na medida de Hilbert-Schmidt.


In [ ]:





# ── Teste rápido de sanidade ──────────────────────────────────────────────────
print("Teste de sanidade: random_density_matrix")
rng_test = np.random.default_rng(seed=0)

for d_test in [2, 4]:
    rho_test = random_density_matrix(d_test, rng=rng_test)
    v = _verify_density_matrix(rho_test)
    status_str = "✓ VÁLIDA" if v["is_valid"] else "✗ INVÁLIDA"
    print(
        f"  d={d_test}: {status_str} | "
        f"herm_err={v['hermitian_error']:.2e} | "
        f"trace_err={v['trace_error']:.2e} | "
        f"lambda_min={v['min_eigenvalue']:.4f}"
    )


## 7. Funções para Construção de Marginais com Sobreposição Garantida

Desenvolvimento de rotinas auxiliares:
- `build_product_marginals`: gera estados produto triviais $\rho_{AB} = \rho_A \otimes \sigma_B$.
- `build_correlated_marginals`: constrói marginais fortemente correlacionadas via purificação e canais quânticos aleatórios.
- `check_overlap_consistency`: avalia a norma da diferença $\|\text{Tr}_B(\rho_{AB}) - \text{Tr}_C(\rho_{AC})\|_F$.


In [ ]:








# ── Teste rápido de sanidade ──────────────────────────────────────────────────
print("Teste de sanidade: build_product_marginals")
rng_t = np.random.default_rng(seed=42)
rho_A_t = random_density_matrix(2, rng=rng_t)
rho_AB_t, rho_AC_t = build_product_marginals(rho_A_t, rng=rng_t)

rho_A_from_AB_t = partial_trace(rho_AB_t, keep=[0], dims=[2, 2])
rho_A_from_AC_t = partial_trace(rho_AC_t, keep=[0], dims=[2, 2])
err_AB = np.linalg.norm(rho_A_from_AB_t - rho_A_t, 'fro')
err_AC = np.linalg.norm(rho_A_from_AC_t - rho_A_t, 'fro')
print(f"  ||Tr_B(rho_AB) - rho_A||_F = {err_AB:.2e}   (esperado < 1e-14)")
print(f"  ||Tr_C(rho_AC) - rho_A||_F = {err_AC:.2e}   (esperado < 1e-14)")

print("\nTeste de sanidade: build_correlated_marginals")
rho_AB_c, rho_AC_c = build_correlated_marginals(rho_A_t, rng=rng_t)
rho_A_from_AB_c = partial_trace(rho_AB_c, keep=[0], dims=[2, 2])
rho_A_from_AC_c = partial_trace(rho_AC_c, keep=[0], dims=[2, 2])
err_AB_c = np.linalg.norm(rho_A_from_AB_c - rho_A_t, 'fro')
err_AC_c = np.linalg.norm(rho_A_from_AC_c - rho_A_t, 'fro')
print(f"  ||Tr_B(rho_AB) - rho_A||_F = {err_AB_c:.2e}")
print(f"  ||Tr_C(rho_AC) - rho_A||_F = {err_AC_c:.2e}")
delta_t, comp_t = check_overlap_consistency(rho_AB_c, rho_AC_c)
print(f"  delta_overlap = {delta_t:.2e},  overlap_compatible = {comp_t}")


## 8. Experimento A: Marginais Completamente Aleatórias ($N = 1000$)

Avaliamos 1000 pares de marginais $\rho_{AB}$ e $\rho_{AC}$ geradas independentemente ao acaso. Como esperado, marginais aleatórias quase nunca satisfazem nem a condição de sobreposição nem a factibilidade do SDP.


In [ ]:

SEED_A = 2024_09_25
N_A    = 1000          # número de realizações
TOL_OVERLAP = 1e-6    # tolerância para consistência na sobreposição
DIMS_3Q = [2, 2, 2]   # 3 qubits

rng_A = np.random.default_rng(seed=SEED_A)
results_A: List[Dict[str, Any]] = []

print(f"Experimento A — Marginais aleatórias (N={N_A} realizações)")
print(f"Tolerância de sobreposição: tol = {TOL_OVERLAP:.0e}")
print("─" * 70)

t_total_A = time.perf_counter()

for i in range(N_A):
    # ── 1. Geração das marginais aleatórias ───────────────────────────────────
    rho_AB = random_density_matrix(4, rng=rng_A)
    rho_AC = random_density_matrix(4, rng=rng_A)

    # ── 2. Teste de consistência na sobreposição A ────────────────────────────
    delta_overlap, overlap_compatible = check_overlap_consistency(
        rho_AB, rho_AC, tol=TOL_OVERLAP
    )

    # ── 3. SDP de representabilidade global (independente do teste de overlap) ─
    marginals = {"AB": rho_AB, "AC": rho_AC}

    t0 = time.perf_counter()
    try:
        sdp, M, b = build_qmp_sdp(marginals, dims=DIMS_3Q)
        res = solve_qmp(sdp, solver="CLARABEL")
        solver_status = res["status"]
        is_feasible   = res["is_feasible"]
    except Exception as exc:
        solver_status = "error"
        is_feasible   = False
    solver_time = time.perf_counter() - t0

    # ── 4. Registro dos resultados ─────────────────────────────────────────────
    results_A.append({
        "experiment":         "A_random",
        "realization":        i,
        "delta_overlap":      delta_overlap,
        "overlap_compatible": overlap_compatible,
        "solver_status":      solver_status,
        "is_feasible":        is_feasible,
        "solver_time":        solver_time,
    })

    # Progresso a cada 100 amostras
    if (i + 1) % 100 == 0:
        n_overlap = sum(r["overlap_compatible"] for r in results_A)
        n_feasible = sum(r["is_feasible"] for r in results_A)
        t_el = time.perf_counter() - t_total_A
        print(
            f"  [{i+1:4d}/{N_A}]  overlap_ok={n_overlap:4d}  "
            f"feasible={n_feasible:4d}  elapsed={t_el:.1f}s"
        )

t_elapsed_A = time.perf_counter() - t_total_A
df_A = pd.DataFrame(results_A)

print("─" * 70)
print(f"Experimento A concluído em {t_elapsed_A:.2f}s")
print(f"  Total amostras:               {len(df_A)}")
print(f"  Overlap compatível:            {df_A['overlap_compatible'].sum()}")
print(f"  Globalmente factível:          {df_A['is_feasible'].sum()}")


## 9. Experimento B: Marginais com Sobreposição Garantida por Construção ($N = 1000$)

Garantimos $Tr_B(\rho_{AB}) = Tr_C(\rho_{AC})$ por construção e testamos a factibilidade SDP para casos produto (`B.prod`) e casos correlacionados (`B.corr`).


In [ ]:

SEED_B = 2024_09_26
N_B    = 1000

rng_B = np.random.default_rng(seed=SEED_B)
results_B: List[Dict[str, Any]] = []

print(f"Experimento B — Marginais com sobreposição garantida (N={N_B} por variante)")
print("─" * 70)

t_total_B = time.perf_counter()

for i in range(N_B):
    # Gera rho_A comum
    rho_A = random_density_matrix(2, rng=rng_B)

    for variant, build_fn, build_kwargs in [
        ("B.prod", build_product_marginals,    {}),
        ("B.corr", build_correlated_marginals, {}),
    ]:
        # ── 1. Constrói as marginais ──────────────────────────────────────────
        rho_AB, rho_AC = build_fn(rho_A, rng=rng_B, **build_kwargs)

        # ── 2. Verificação numérica das marginais em A ──────────────────────
        rho_A_from_AB = partial_trace(rho_AB, keep=[0], dims=[2, 2])
        rho_A_from_AC = partial_trace(rho_AC, keep=[0], dims=[2, 2])
        err_AB = float(np.linalg.norm(rho_A_from_AB - rho_A, 'fro'))
        err_AC = float(np.linalg.norm(rho_A_from_AC - rho_A, 'fro'))

        # ── 3. Teste de sobreposição ────────────────────────────────────────
        delta_overlap, overlap_compatible = check_overlap_consistency(
            rho_AB, rho_AC, tol=TOL_OVERLAP
        )

        # ── 4. SDP de representabilidade global ─────────────────────────────
        marginals = {"AB": rho_AB, "AC": rho_AC}

        t0 = time.perf_counter()
        try:
            sdp, M, b = build_qmp_sdp(marginals, dims=DIMS_3Q)
            res = solve_qmp(sdp, solver="CLARABEL")
            solver_status = res["status"]
            is_feasible   = res["is_feasible"]
        except Exception as exc:
            solver_status = "error"
            is_feasible   = False
        solver_time = time.perf_counter() - t0

        # ── 5. Registro ──────────────────────────────────────────────────────
        results_B.append({
            "experiment":         variant,
            "realization":        i,
            "delta_overlap":      delta_overlap,
            "overlap_compatible": overlap_compatible,
            "marginal_err_AB":    err_AB,
            "marginal_err_AC":    err_AC,
            "solver_status":      solver_status,
            "is_feasible":        is_feasible,
            "solver_time":        solver_time,
        })

    # Progresso
    if (i + 1) % 200 == 0:
        n_prod_feas = sum(
            r["is_feasible"] for r in results_B if r["experiment"] == "B.prod"
        )
        n_corr_feas = sum(
            r["is_feasible"] for r in results_B if r["experiment"] == "B.corr"
        )
        t_el = time.perf_counter() - t_total_B
        print(
            f"  [{i+1:4d}/{N_B}]  prod_feasible={n_prod_feas:4d}  "
            f"corr_feasible={n_corr_feas:4d}  elapsed={t_el:.1f}s"
        )

t_elapsed_B = time.perf_counter() - t_total_B
df_B = pd.DataFrame(results_B)

print("─" * 70)
print(f"Experimento B concluído em {t_elapsed_B:.2f}s")
for variant in ["B.prod", "B.corr"]:
    sub = df_B[df_B["experiment"] == variant]
    print(f"  Variante {variant}:")
    print(f"    Total amostras:     {len(sub)}")
    print(f"    Overlap compatível: {sub['overlap_compatible'].sum()} ({100*sub['overlap_compatible'].mean():.1f}%)")
    print(f"    Globalmente fact.:  {sub['is_feasible'].sum()} ({100*sub['is_feasible'].mean():.1f}%)")
    print(f"    Err AB médio:       {sub['marginal_err_AB'].mean():.2e}")
    print(f"    Err AC médio:       {sub['marginal_err_AC'].mean():.2e}")


## 11. Estatísticas Consolidadas dos Experimentos

Agregação dos resultados em um DataFrame do `pandas` e exibição da tabela comparativa dos quatro grupos de experimentos (A, B.prod, B.corr, Bell).


In [ ]:

# Combina todos os resultados em um único DataFrame
df_all = pd.concat([df_A, df_B, df_bell], ignore_index=True)

# Garante tipos corretos
df_all["overlap_compatible"] = df_all["overlap_compatible"].astype(bool)
df_all["is_feasible"]        = df_all["is_feasible"].astype(bool)




print("RESUMO ESTATÍSTICO COMPLETO")
stats_summary = []

for exp_name in df_all["experiment"].unique():
    sub = df_all[df_all["experiment"] == exp_name]
    s = print_experiment_stats(sub, exp_name)
    stats_summary.append(s)

# Tabela resumo
print("\n" + "=" * 90)
print("TABELA RESUMO")
print("=" * 90)
df_stats = pd.DataFrame(stats_summary)
print(df_stats.to_string(index=False))

# ── Análise focada: sobreposição ≠ compatibilidade global ─────────────────────
print("\n" + "=" * 65)
print("CONCLUSÃO PRINCIPAL: overlap ≠ representabilidade global")
print("=" * 65)

for row in stats_summary:
    if row["n_overlap_infeasible"] > 0:
        pct = 100 * row["n_overlap_infeasible"] / row["n_overlap"] if row["n_overlap"] > 0 else 0
        print(
            f"  [{row['label']}] {row['n_overlap_infeasible']} par(es) com "
            f"overlap ✓ mas SDP infactível ({pct:.1f}% dos com overlap)"
        )


## 12. Dashboard de Visualizações Integradas (6 Painéis Gráficos)

Painel analítico contendo:
1. Histograma de erro de sobreposição em marginais aleatórias.
2. Comparativo de taxa de compatibilidade por experimento.
3. Distribuição de autovalores mínimos.
4. Análise dos 4 quadrantes lógicos (Overlap vs SDP Factível).
5. Comparativo $B.prod$ vs $B.corr$.
6. Distribuição do tempo de execução do solver **CLARABEL**.


In [ ]:

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = {
    "overlap_true":  "#2ecc71",   # verde
    "overlap_false": "#e74c3c",   # vermelho
    "feasible":      "#3498db",   # azul
    "infeasible":    "#e67e22",   # laranja
    "neutral":       "#95a5a6",   # cinza
}

fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    "Compatibilidade de Marginais Quânticas para 3 Qubits\n"
    r"Overlap Consistency $\neq$ Global Representability",
    fontsize=15, fontweight="bold", y=0.98
)

# ─────────────────────────────────────────────────────────────────────────────
# Gráfico 1: Histograma de delta_overlap — Experimento A (marginais aleatórias)
# ─────────────────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)

data_A_delta = df_A["delta_overlap"].values
ax1.hist(
    data_A_delta, bins=50,
    color=COLORS["neutral"], edgecolor="white", linewidth=0.5
)
ax1.axvline(
    TOL_OVERLAP, color="red", linestyle="--", linewidth=1.5,
    label=f"tol = {TOL_OVERLAP:.0e}"
)
ax1.set_xlabel(r"$\|\mathrm{Tr}_B(\rho_{AB}) - \mathrm{Tr}_C(\rho_{AC})\|_F$", fontsize=11)
ax1.set_ylabel("Frequência", fontsize=11)
ax1.set_title("Exp. A: Histograma de delta_overlap\n(marginais aleatórias)", fontsize=11)
ax1.legend(fontsize=10)

n_left = (data_A_delta < TOL_OVERLAP).sum()
ax1.text(
    0.02, 0.97,
    f"{n_left} pares com\noverlap < tol",
    transform=ax1.transAxes, va="top", fontsize=9,
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.7)
)

# ─────────────────────────────────────────────────────────────────────────────
# Gráfico 2: Barras — overlap_compatible vs is_feasible por experimento
# ─────────────────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(2, 3, 2)

exp_labels = []
pct_overlap = []
pct_feasible = []

for exp_name in ["A_random", "B.prod", "B.corr"]:
    sub = df_all[df_all["experiment"] == exp_name]
    exp_labels.append(exp_name)
    pct_overlap.append(100 * sub["overlap_compatible"].mean())
    pct_feasible.append(100 * sub["is_feasible"].mean())

x_pos = np.arange(len(exp_labels))
width = 0.35

bars1 = ax2.bar(
    x_pos - width/2, pct_overlap, width,
    label="Overlap compatível", color=COLORS["overlap_true"], alpha=0.85
)
bars2 = ax2.bar(
    x_pos + width/2, pct_feasible, width,
    label="SDP factível", color=COLORS["feasible"], alpha=0.85
)

for bar in bars1:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 1, f"{h:.1f}%",
             ha="center", va="bottom", fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 1, f"{h:.1f}%",
             ha="center", va="bottom", fontsize=8)

ax2.set_xticks(x_pos)
ax2.set_xticklabels(exp_labels, fontsize=10)
ax2.set_ylabel("% de amostras", fontsize=11)
ax2.set_title("Overlap vs Factibilidade\npor experimento", fontsize=11)
ax2.set_ylim(0, 115)
ax2.legend(fontsize=9)

# ─────────────────────────────────────────────────────────────────────────────
# Gráfico 3: Distribuição de delta_overlap — casos compatíveis vs incompatíveis
#            (Exp. B.corr — mais interessante)
# ─────────────────────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)

df_Bcorr = df_all[df_all["experiment"] == "B.corr"]
delta_Bcorr_feasible   = df_Bcorr[df_Bcorr["is_feasible"] == True]["delta_overlap"].values
delta_Bcorr_infeasible = df_Bcorr[df_Bcorr["is_feasible"] == False]["delta_overlap"].values

bins_bc = np.linspace(0, max(df_Bcorr["delta_overlap"].max(), 1e-10), 40)

if len(delta_Bcorr_feasible) > 0:
    ax3.hist(
        delta_Bcorr_feasible, bins=bins_bc, alpha=0.6,
        label=f"SDP factível (n={len(delta_Bcorr_feasible)})",
        color=COLORS["feasible"]
    )
if len(delta_Bcorr_infeasible) > 0:
    ax3.hist(
        delta_Bcorr_infeasible, bins=bins_bc, alpha=0.6,
        label=f"SDP infactível (n={len(delta_Bcorr_infeasible)})",
        color=COLORS["infeasible"]
    )

ax3.axvline(TOL_OVERLAP, color="red", linestyle="--", linewidth=1.5, label="tol")
ax3.set_xlabel(r"$\delta_{\mathrm{overlap}}$", fontsize=11)
ax3.set_ylabel("Frequência", fontsize=11)
ax3.set_title(
    "Exp. B.corr: delta_overlap\nfactível vs infactível", fontsize=11
)
ax3.legend(fontsize=9)

# ─────────────────────────────────────────────────────────────────────────────
# Gráfico 4: Quadrantes lógicos — overlap × SDP
# ─────────────────────────────────────────────────────────────────────────────
ax4 = fig.add_subplot(2, 3, 4)

quadrant_data = {}
for exp_name in ["A_random", "B.prod", "B.corr"]:
    sub = df_all[df_all["experiment"] == exp_name]
    quadrant_data[exp_name] = {
        "OV=T, SDP=T":  int((sub["overlap_compatible"] &  sub["is_feasible"]).sum()),
        "OV=T, SDP=F":  int((sub["overlap_compatible"] & ~sub["is_feasible"]).sum()),
        "OV=F, SDP=T":  int((~sub["overlap_compatible"] &  sub["is_feasible"]).sum()),
        "OV=F, SDP=F":  int((~sub["overlap_compatible"] & ~sub["is_feasible"]).sum()),
    }

quad_labels = ["OV=T\nSDP=T", "OV=T\nSDP=F", "OV=F\nSDP=T", "OV=F\nSDP=F"]
quad_colors = [
    COLORS["feasible"],
    COLORS["infeasible"],
    COLORS["neutral"],
    COLORS["overlap_false"],
]

exp_names_q = ["A_random", "B.prod", "B.corr"]
x_q = np.arange(len(quad_labels))
bar_width = 0.25

for k, exp_name in enumerate(exp_names_q):
    vals = [quadrant_data[exp_name][q.replace("\n", "=".join(["OV", "SDP"]))
                                     .replace("\n", ", ")]
            for q in ["OV=T, SDP=T", "OV=T, SDP=F", "OV=F, SDP=T", "OV=F, SDP=F"]]
    ax4.bar(
        x_q + k * bar_width - bar_width, vals, bar_width,
        label=exp_name, alpha=0.8,
        color=["#2ecc71", "#e67e22", "#3498db", "#e74c3c"][0],  # placeholder color
    )

# Redesenha com cores corretas por quadrante
ax4.cla()
offsets = [-bar_width, 0, bar_width]
exp_plot_colors = ["#2ecc71", "#3498db", "#e67e22"]

for k, (exp_name, color) in enumerate(zip(exp_names_q, exp_plot_colors)):
    vals = [
        quadrant_data[exp_name]["OV=T, SDP=T"],
        quadrant_data[exp_name]["OV=T, SDP=F"],
        quadrant_data[exp_name]["OV=F, SDP=T"],
        quadrant_data[exp_name]["OV=F, SDP=F"],
    ]
    bars_q = ax4.bar(
        x_q + offsets[k], vals, bar_width,
        label=exp_name, color=color, alpha=0.82, edgecolor="white"
    )
    for bar, v in zip(bars_q, vals):
        if v > 0:
            ax4.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
                str(v), ha="center", va="bottom", fontsize=7
            )

ax4.set_xticks(x_q)
ax4.set_xticklabels(quad_labels, fontsize=10)
ax4.set_ylabel("Número de amostras", fontsize=11)
ax4.set_title(
    "Quadrantes: Overlap × Factibilidade SDP",
    fontsize=11
)
ax4.legend(fontsize=9, loc="upper right")

# Adiciona anotação de destaque no quadrante crítico OV=T, SDP=F
ax4.annotate(
    "← Região crítica:\noverlap ✓ mas\nextensão global ✗",
    xy=(1, max(quadrant_data[exp]["OV=T, SDP=F"] for exp in exp_names_q) + 5),
    xytext=(2, max(quadrant_data[exp]["OV=T, SDP=F"] for exp in exp_names_q) + 50),
    arrowprops=dict(arrowstyle="->", color="black"),
    fontsize=9, color="darkred"
)

# ─────────────────────────────────────────────────────────────────────────────
# Gráfico 5: Histograma de delta_overlap — Experimentos B.prod e B.corr
# ─────────────────────────────────────────────────────────────────────────────
ax5 = fig.add_subplot(2, 3, 5)

for exp_name, color, ls in [
    ("B.prod", "#3498db", "-"),
    ("B.corr", "#e67e22", "--"),
]:
    sub = df_all[df_all["experiment"] == exp_name]
    ax5.hist(
        sub["delta_overlap"].values,
        bins=50, alpha=0.6, label=exp_name,
        color=color, edgecolor="white", linewidth=0.5
    )

ax5.axvline(TOL_OVERLAP, color="red", linestyle="--", linewidth=1.5, label=f"tol={TOL_OVERLAP:.0e}")
ax5.set_xlabel(r"$\delta_{\mathrm{overlap}}$", fontsize=11)
ax5.set_ylabel("Frequência", fontsize=11)
ax5.set_title("Exp. B: delta_overlap\n(produto vs correlacionado)", fontsize=11)
ax5.legend(fontsize=9)
ax5.set_xscale("symlog", linthresh=1e-10)

# ─────────────────────────────────────────────────────────────────────────────
# Gráfico 6: Tempo de resolução do SDP por experimento (boxplot)
# ─────────────────────────────────────────────────────────────────────────────
ax6 = fig.add_subplot(2, 3, 6)

exp_for_time = ["A_random", "B.prod", "B.corr"]
time_data = [
    df_all[df_all["experiment"] == exp]["solver_time"].values
    for exp in exp_for_time
]

bp = ax6.boxplot(
    time_data, label=exp_for_time,
    patch_artist=True, notch=False,
    medianprops=dict(color="black", linewidth=2)
)
box_colors = ["#2ecc71", "#3498db", "#e67e22"]
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax6.set_ylabel("Tempo de resolução (s)", fontsize=11)
ax6.set_title("Tempo do Solver CLARABEL\npor experimento", fontsize=11)
ax6.set_yscale("log")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("experimento_compatibilidade_marginais.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✓ Figura salva: experimento_compatibilidade_marginais.png")


## 13. Conclusões e Síntese dos Resultados

Resumo final dos achados experimentais e salvamento dos relatórios.


In [ ]:

print("=" * 70)
print("RESUMO DO EXPERIMENTO")
print("=" * 70)
print("""
Este notebook implementou um estudo numérico da região de compatibilidade
de marginais quânticas para sistemas de 3 qubits.

FUNÇÕES REUTILIZADAS (de inicial_teste.ipynb, sem modificações):
  - ketbra(psi)                   → projetor |psi><psi|
  - tensor(*args)                 → produto de Kronecker sequencial
  - partial_trace(rho, keep, dims)→ traço parcial via einsum
  - base_hermitiana(d)            → base de Herm(d)
  - embed_operator(O, sys, dim)   → embedding de operador local
  - parse_subsystem_key(key)      → normalização de chaves de subsistema
  - build_qmp_sdp(marginals, dims)→ montagem do SDP (SDPLab)
  - solve_qmp(sdp, solver)        → solução do SDP via CLARABEL

NOVAS FUNÇÕES (específicas deste experimento):
  - random_density_matrix(d, rng)         → estado aleatório (Ginibre)
  - build_product_marginals(rho_A, rng)   → rho_AB=rho_A⊗sigma_B (trivial)
  - build_correlated_marginals(rho_A,rng) → purificação + canal aleatório
  - check_overlap_consistency(rho_AB, rho_AC, tol)
  - print_experiment_stats(df, label)     → estatísticas consolidadas

EXPERIMENTOS:
  - Exp. A  (N=1000): marginais completamente aleatórias
  - Exp. B.prod (N=1000): rho_AB = rho_A ⊗ sigma_B (sobreposição garantida)
  - Exp. B.corr (N=1000): purificação correlacionada (sobreposição garantida)
  - Bell (N=1): caso de validação — overlap ✓ mas SDP infactível

RESULTADO PRINCIPAL:
  A consistência na sobreposição (Tr_B(rho_AB) = Tr_C(rho_AC)) é condição
  NECESSÁRIA mas NÃO SUFICIENTE para existência de extensão global rho_ABC.
  O caso de Bell e os casos B.corr demonstram isso numericamente.
""")

# Exibe o DataFrame final com todos os resultados
print("Primeiras 5 linhas do DataFrame completo:")
print(df_all[["experiment", "delta_overlap", "overlap_compatible",
              "solver_status", "is_feasible", "solver_time"]].head())
